# Week 3 — Our First Prediction Model (Stimulus Only)

So far we've *looked* at data. Now we **predict**. The big question of the whole project is:

> Given information about a trial, can we predict how each neuron will respond?

This week we build the **baseline**: predict neural responses using **only the image** the mouse saw. Next week we'll add behavior and see if it helps. To make a fair comparison later, we set up the pipeline carefully today.

**New ideas this week:**
1. **Train/test split** — learn on some trials, test on *unseen* ones.
2. **Features** — turning a big image into a manageable list of numbers (PCA).
3. **Ridge regression** — a simple, honest prediction model.

> Runs in Colab, no GPU needed.

## 1. Setup: get the data and load it

Same mouse as Week 2. (If your Colab session reset, this re-downloads it.) We load the **images**, the **responses**, and the train/validation/test **labels**.

In [ ]:
FILENAME = "static21067-10-18-GrayImageNet-94c6ff995dac583098847cfecd43e7b6"
URL = f"https://gin.g-node.org/cajal/Sensorium2022/raw/master/{FILENAME}.zip"

import os
if not os.path.exists(FILENAME):
    print("Downloading (~424 MB)...")
    !wget -q --show-progress -O mouse.zip "$URL"
    !unzip -q -o mouse.zip
print("✅ Data ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = os.path.join(FILENAME, "data")
n_trials  = len(os.listdir(f"{DATA_DIR}/responses"))
n_neurons = np.load(f"{DATA_DIR}/responses/0.npy").shape[0]

# We shrink each 144x256 image to 36x64 by averaging 4x4 blocks.
# (Full-size images are too big to handle directly; the shrunk version keeps the gist.)
def downsample(img):
    a = img[0]                                   # (144, 256)
    return a.reshape(36, 4, 64, 4).mean(axis=(1, 3)).ravel()   # -> 2304 numbers

print(f"Loading {n_trials} trials...")
X_image = np.zeros((n_trials, 36*64), dtype=np.float32)   # shrunk images
R       = np.zeros((n_trials, n_neurons), dtype=np.float32)  # responses
for i in range(n_trials):
    X_image[i] = downsample(np.load(f"{DATA_DIR}/images/{i}.npy"))
    R[i]       = np.load(f"{DATA_DIR}/responses/{i}.npy")

tiers = np.load(f"{FILENAME}/meta/trials/tiers.npy", allow_pickle=True)
print("Done. Image table:", X_image.shape, "| Response table:", R.shape)

## 2. Train / test split — the most important idea in machine learning

If we test a model on the same data it learned from, it can just *memorize* and look great — but be useless on anything new. So we split trials into:
- **Training** trials — the model learns from these.
- **Test** trials — held back; we only use them to *check* the model on data it never saw.

The dataset already comes with these labels (`train`, `validation`, `test`). We'll train on `train`, and test on the rest.

In [ ]:
is_train = (tiers == "train")
is_test  = (tiers == "validation") | (tiers == "test")

print(f"Training trials: {is_train.sum()}")
print(f"Test trials:     {is_test.sum()}")

## 3. Turn images into features with PCA

Even shrunk, each image is 2,304 numbers — many of them redundant (neighboring pixels look alike). **PCA** (Principal Component Analysis) finds a smaller set of numbers that still captures most of what makes images different. We'll squeeze 2,304 numbers down to **150**.

**Important:** we *learn* the PCA summary from the **training images only**, then apply it to all images. (Never let the model peek at test data while learning — that would be cheating.)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1) Put pixels on a common scale (fit on training data only)
scaler = StandardScaler().fit(X_image[is_train])
X_scaled = scaler.transform(X_image)

# 2) PCA down to 150 features (fit on training data only)
pca = PCA(n_components=150, random_state=0).fit(X_scaled[is_train])
stim_features = pca.transform(X_scaled)

print("Image features shape:", stim_features.shape, "(trials x 150 features)")
print(f"These 150 features capture {pca.explained_variance_ratio_.sum()*100:.0f}% of the image variation.")

## 4. The model: Ridge regression

**Ridge regression** predicts each neuron's response as a weighted sum of the image features. The word "ridge" refers to a bit of built-in caution (`alpha`) that stops the model from over-trusting any single feature — which helps it work on new data.

One `Ridge` model can predict **all 8,372 neurons at once**. We `fit` it on the training trials.

In [ ]:
from sklearn.linear_model import Ridge

ALPHA = 1000.0   # the 'caution' knob; higher = more cautious

model = Ridge(alpha=ALPHA)
model.fit(stim_features[is_train], R[is_train])
print("Model trained on", is_train.sum(), "trials.")

## 5. How good is it? Test on unseen trials

We ask the model to predict responses for the **test** trials it never saw, then measure — for each neuron — the **correlation** between predicted and actual responses. Correlation near 1 = great; near 0 = no better than guessing the average.

In [ ]:
def per_neuron_correlation(predicted, actual):
    """Correlation between predicted and actual response, for each neuron."""
    p = predicted - predicted.mean(axis=0)
    a = actual - actual.mean(axis=0)
    return (p*a).sum(axis=0) / (np.sqrt((p**2).sum(axis=0) * (a**2).sum(axis=0)) + 1e-12)

predicted_test = model.predict(stim_features[is_test])
baseline_corr  = per_neuron_correlation(predicted_test, R[is_test])

print(f"BASELINE (stimulus only) mean test correlation: {np.nanmean(baseline_corr):.3f}")
print(f"Best-predicted neuron:  {np.nanmax(baseline_corr):.3f}")

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(baseline_corr, bins=50)
plt.axvline(np.nanmean(baseline_corr), color="red", linestyle="--",
            label=f"mean = {np.nanmean(baseline_corr):.3f}")
plt.xlabel("test correlation (predicted vs actual)")
plt.ylabel("number of neurons")
plt.title("How well the stimulus-only model predicts each neuron")
plt.legend(); plt.show()

## 6. See a prediction

Let's look at our best-predicted neuron and plot its predicted vs actual responses across test trials.

In [ ]:
best = int(np.nanargmax(baseline_corr))
plt.figure(figsize=(5,5))
plt.scatter(R[is_test][:, best], predicted_test[:, best], alpha=0.3, s=10)
plt.xlabel("actual response"); plt.ylabel("predicted response")
plt.title(f"Neuron {best} (correlation = {baseline_corr[best]:.2f})")
lims = [R[is_test][:,best].min(), R[is_test][:,best].max()]
plt.plot(lims, lims, "k--", linewidth=0.8)   # perfect-prediction line
plt.show()

## 7. An honest reflection

Our baseline correlation is **modest** (around 0.12–0.13). That's expected and okay:
- Visual cortex neurons respond to images in **complicated, non-linear** ways. A simple linear model + shrunk images can only capture so much. (State-of-the-art deep networks do better — that could be a future extension.)
- But we don't need a perfect model. We need a **fair, consistent baseline** so that next week we can ask a clean question: *does adding behavior improve it?*

**We have that baseline now.** Let's save everything so Week 4 continues seamlessly.

In [ ]:
# Also load behavior now, so Week 4 has everything in one file.
B = np.zeros((n_trials, 3))
for i in range(n_trials):
    B[i] = np.load(f"{DATA_DIR}/behavior/{i}.npy").ravel()

np.savez_compressed("week3_prepared.npz",
                    stim_features=stim_features, responses=R, behavior=B,
                    tiers=tiers, baseline_corr=baseline_corr)
print("Saved week3_prepared.npz  (Week 4 will load this)")
# To keep it across sessions, uncomment:
# from google.colab import files; files.download('week3_prepared.npz')

## Exercises

1. Change `ALPHA` in Section 4 to `10` and to `100000`, re-run Sections 4–5. How does the mean test correlation change? (This shows why the 'caution' knob matters.)
2. Change `n_components=150` in Section 3 to `20`. Fewer image features — does prediction get worse? By how much?
3. How many neurons are predicted with correlation above 0.3? Hint: `np.mean(baseline_corr > 0.3)`.

_Your notes:_



## Recap

- You built your **first prediction model** and evaluated it the right way — on **held-out test trials**.
- You learned **train/test split**, **PCA features**, and **ridge regression**.
- You have a fair **stimulus-only baseline** (~0.13 mean correlation).

### Next week (Week 4) — the payoff
We add the mouse's **behavior** (pupil, running) to the model and test our core hypothesis: **does knowing behavior help predict the brain?** (Spoiler from our Week 2 peek: it should — especially for arousal-sensitive neurons.)

You just trained a real model of a real brain. 🧠⚡